# Smart Farming — Notebook 1: Save model-service files to Google Drive

**Run this notebook first.** It copies the repository's model service, advisory knowledge base, model assets (if tracked in the repository), and relevant configuration files into `MyDrive/SmartFarming` without deleting existing Drive files.

This notebook does not run inference. It does not magically copy model weights that exist only on your local PC or are excluded from Git/Git LFS; upload those to Drive separately if your service requires them.

After the sync, set up the Cloudflare Worker using the included `cloudflare_worker.js` file:
1. Create a Cloudflare KV namespace and bind it to the Worker as `UPSTREAMS`.
2. Add a Worker secret named `GATEWAY_API_KEY`.
3. Add Colab Secrets in the later notebooks: `CF_API_TOKEN`, `CF_ACCOUNT_ID`, `CF_KV_NAMESPACE_ID`, `CF_WORKER_BASE_URL`, `GATEWAY_API_KEY`, and `HF_TOKEN` (HF token may be needed for gated model downloads).
4. The Cloudflare API token needs **Workers KV Storage: Edit** for the target account. The gateway key is separate from the Cloudflare API token.


## 1. Mount Drive and sync repository files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, json
REPO = "/content/Smart-Farming"
DRIVE_ROOT = "/content/drive/MyDrive/SmartFarming"

if os.path.isdir(REPO):
    shutil.rmtree(REPO)

subprocess.run([
    "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
    "https://github.com/manthan2876/Smart-Farming.git", REPO
], check=True)

subprocess.run(
    ["git", "-C", REPO, "sparse-checkout", "set", "model_service", "data", "models"],
    check=True
)

os.makedirs(DRIVE_ROOT, exist_ok=True)

def copy_tree_merge(source, destination):
    if not os.path.isdir(source):
        print(f"Not present in repository: {source}")
        return
    os.makedirs(destination, exist_ok=True)
    for name in os.listdir(source):
        s, d = os.path.join(source, name), os.path.join(destination, name)
        if os.path.isdir(s) and not os.path.islink(s):
            copy_tree_merge(s, d)
        else:
            shutil.copy2(s, d)

copy_tree_merge(os.path.join(REPO, "model_service"), DRIVE_ROOT)
copy_tree_merge(os.path.join(REPO, "models"), os.path.join(DRIVE_ROOT, "models"))
copy_tree_merge(os.path.join(REPO, "data"), os.path.join(DRIVE_ROOT, "data"))

# Keep a top-level KB path used by advisory_server.py when the repo provides it.
for candidate in [
    os.path.join(REPO, "data", "advisory_kb.json"),
    os.path.join(REPO, "model_service", "advisory_kb.json")
]:
    if os.path.isfile(candidate):
        shutil.copy2(candidate, os.path.join(DRIVE_ROOT, "advisory_kb.json"))
        print("Copied advisory knowledge base from:", candidate)
        break

print("\nDrive sync finished:", DRIVE_ROOT)
print("Top-level contents:", sorted(os.listdir(DRIVE_ROOT)))


## 2. Verify copied service files and model assets

In [ ]:
import os
for root, dirs, files in os.walk(DRIVE_ROOT):
    rel = os.path.relpath(root, DRIVE_ROOT)
    for name in files:
        path = os.path.join(root, name)
        print(f"{os.path.join(rel, name)}  ({os.path.getsize(path):,} bytes)")


## 3. Cloudflare Worker setup

Deploy `cloudflare_worker.js` as a Worker, then:

- Create a KV namespace (for example `SMART_FARMING_UPSTREAMS`) and bind it to the Worker with binding name **`UPSTREAMS`**.
- Add a Worker secret named **`GATEWAY_API_KEY`**. Use a long random value.
- Keep the `workers.dev` URL as `CF_WORKER_BASE_URL`, e.g. `https://smart-farming-gateway.YOUR-SUBDOMAIN.workers.dev`.
- In Colab Secrets, add `CF_API_TOKEN`, `CF_ACCOUNT_ID`, and `CF_KV_NAMESPACE_ID`. The token needs permission to edit KV values in the selected namespace/account.
- Keep `GATEWAY_API_KEY` in Colab Secrets too, with exactly the same value as the Worker secret.
- Add `HF_TOKEN` to Colab Secrets if the models require Hugging Face authentication.

The next two notebooks update KV keys automatically after creating their new Quick Tunnel URLs. The app's base URL stays fixed; Colab still must be running for requests to succeed.


## 4. Optional: save the Worker source beside this notebook

In [ ]:
worker_source = r'''/**
 * Smart Farming stable gateway.
 * Bind a Cloudflare KV namespace to this Worker using the binding name UPSTREAMS.
 * Create a Worker secret named GATEWAY_API_KEY.
 * KV keys: VISION_URL, TRANSLATION_URL, ADVISORY_URL.
 *
 * Routes:
 *   /vision/*      -> VISION_URL + suffix
 *   /translation/* -> TRANSLATION_URL + suffix
 *   /advisory/*    -> ADVISORY_URL + suffix
 */
export default {
  async fetch(request, env) {
    const url = new URL(request.url);

    if (request.method === "OPTIONS") {
      return new Response(null, {
        status: 204,
        headers: {
          "Access-Control-Allow-Origin": "*",
          "Access-Control-Allow-Methods": "GET, POST, PUT, PATCH, DELETE, OPTIONS",
          "Access-Control-Allow-Headers": "Content-Type, Authorization",
          "Access-Control-Max-Age": "86400"
        }
      });
    }

    if (!env.UPSTREAMS || !env.GATEWAY_API_KEY) {
      return Response.json(
        { error: "Worker is missing UPSTREAMS KV binding or GATEWAY_API_KEY secret" },
        { status: 500 }
      );
    }

    if (request.headers.get("Authorization") !== `Bearer ${env.GATEWAY_API_KEY}`) {
      return Response.json({ error: "Unauthorized" }, { status: 401 });
    }

    const routes = [
      { prefix: "/vision", key: "VISION_URL" },
      { prefix: "/translation", key: "TRANSLATION_URL" },
      { prefix: "/advisory", key: "ADVISORY_URL" },
      { prefix: "/tts", key: "TTS_URL" }
    ];
    const route = routes.find(r =>
      url.pathname === r.prefix || url.pathname.startsWith(r.prefix + "/")
    );

    if (!route) {
      return Response.json({ error: "Unknown service route" }, { status: 404 });
    }

    const origin = await env.UPSTREAMS.get(route.key);
    if (!origin) {
      return Response.json({ error: `Upstream ${route.key} is not configured yet` }, { status: 503 });
    }

    let target;
    try {
      target = new URL(origin);
      const suffix = url.pathname.slice(route.prefix.length);
      target.pathname = target.pathname.replace(/\/+$/, "") + (suffix || "/");
      target.search = url.search;
    } catch {
      return Response.json({ error: `Invalid upstream URL stored for ${route.key}` }, { status: 500 });
    }

    try {
      const headers = new Headers(request.headers);
      headers.delete("Host");
      // Authorization is gateway-only; don't expose the shared gateway secret upstream.
      headers.delete("Authorization");
      const init = {
        method: request.method,
        headers,
        redirect: "manual"
      };
      if (!["GET", "HEAD"].includes(request.method)) init.body = request.body;
      const upstream = await fetch(target.toString(), init);
      const outHeaders = new Headers(upstream.headers);
      outHeaders.set("Access-Control-Allow-Origin", "*");
      outHeaders.set("Access-Control-Allow-Headers", "Content-Type, Authorization");
      outHeaders.set("Access-Control-Allow-Methods", "GET, POST, PUT, PATCH, DELETE, OPTIONS");
      return new Response(upstream.body, {
        status: upstream.status,
        statusText: upstream.statusText,
        headers: outHeaders
      });
    } catch (error) {
      return Response.json(
        { error: "Upstream service unavailable", detail: "The Colab runtime or Quick Tunnel may be offline." },
        { status: 502 }
      );
    }
  }
};
'''
worker_file = "/content/drive/MyDrive/SmartFarming/cloudflare_worker.js"
with open(worker_file, "w", encoding="utf-8") as f:
    f.write(worker_source)
print("Worker source saved to:", worker_file)
